Plotting for Salmon align, using bam files from STAR

Load packages

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px

from sklearn.decomposition import PCA

PCA

Load datasets created in R

In [2]:
norm_counts = pd.read_csv("C:/Users/Sebas/OneDrive/Dokument/Master courses/MASTER THESIS/R project-Master Thesis/salmon_alignment_normalized_counts_transcript_consistent_new_filtering_genotype_controlled_fixed_2.csv", index_col=0)
norm_counts

,ERR12356072,ERR12383247,ERR12383248,ERR12383249,ERR12383250,ERR12383251,ERR12383252,ERR12383253,ERR12383254,ERR12383255,...,ERR12383308,ERR12383309,ERR12383310,ERR12383311,ERR12383312,ERR12383313,ERR12383314,ERR12383315,ERR12383316,ERR12383317
g2.t1,8.179971,8.314462,8.366862,9.180367,8.768474,9.484972,8.072014,8.311687,8.363032,9.002976,...,8.644292,8.412007,8.705738,7.990132,8.106638,8.489611,8.127510,8.538823,8.598385,9.101814
g3.t1,7.519554,7.378705,7.770764,7.527891,8.153554,7.805290,7.333889,7.717002,7.519212,7.737365,...,7.025487,7.345657,6.938024,6.735438,6.861971,6.929487,6.941810,7.737271,8.099690,7.749338
g4.t1,7.933444,7.785521,7.718122,7.497031,7.414156,7.368143,7.787534,7.699406,7.963289,7.445777,...,7.673908,7.675254,7.610171,7.874893,7.830012,7.728258,7.720900,7.437437,7.548877,7.073386
g6.t1,8.477737,8.804940,8.402280,8.042027,8.279223,8.147187,8.798089,8.452804,8.587241,8.228194,...,8.065112,8.147488,8.427252,8.550551,8.762452,8.720101,8.736995,8.139997,7.995304,7.846206
g7.t1,7.050148,7.204487,7.200538,6.764972,6.860239,6.808397,7.374628,7.388358,7.223742,6.624187,...,7.239564,6.758273,6.690006,7.278719,7.081058,7.128189,7.279677,7.109924,7.113527,6.958390
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
g34843.t1,6.519780,6.671473,6.557459,6.728710,6.286897,6.783712,6.603370,6.508287,6.660666,6.917068,...,6.477838,6.239107,6.409367,6.631086,6.147886,6.192146,6.503289,6.635670,6.717653,6.653907
g34884.t1,5.978449,6.138251,5.978449,5.978449,5.978449,6.439258,6.134509,5.978449,5.978449,5.978449,...,6.563129,6.417657,6.137641,6.158407,6.097446,6.285033,5.978449,6.324831,6.164684,6.446313
g34922.t1,6.283874,6.354691,6.340268,6.323012,6.290250,6.657408,6.342898,5.978449,6.214265,5.978449,...,6.334335,6.316861,6.350271,6.428668,6.153456,6.304675,6.262332,6.482361,5.978449,6.295360
g35167.t1,6.176934,6.307844,6.592178,6.617075,5.978449,6.645511,6.321622,6.376566,6.592267,6.330279,...,6.434802,6.194932,6.346557,6.342174,6.250941,6.227461,6.260634,6.677481,6.618157,6.855034


In [3]:
#Metadata
metadata = pd.read_csv("C:/Users/Sebas/OneDrive/Dokument/Master courses/MASTER THESIS/R project-Master Thesis/dominance_meta_corrected_outlier_corrected.csv", sep =";")

#make sure order matches norm_counts
metadata = metadata.set_index('Run')
metadata = metadata.reindex(norm_counts.columns)
# Now Run is the index, and metadata is aligned with counts
print("Metadata index (samples):", metadata.index[:5].tolist())
print("Counts columns:", norm_counts.columns[:5].tolist())
print("Match?", all(metadata.index == norm_counts.columns))
metadata

Metadata index (samples): ['ERR12356072', 'ERR12383247', 'ERR12383248', 'ERR12383249', 'ERR12383250']
Counts columns: ['ERR12356072', 'ERR12383247', 'ERR12383248', 'ERR12383249', 'ERR12383250']
Match? True


,Bases,BioProject,BioSample,Experiment,sample_name_incorrect,TF ID,Sample ID,Cross,Family,Sex,Reciprocal Genotype,Genotype,Pairwise Cross,original_fastq_name_R1,original_fastq_name_R2,Notes
ERR12356072,21027960416,PRJEB70958,SAMEA114860228,ERX11733017,Sample 1 males genotype BA heterozygote,TF2581-10-e3,10e 3,13:20 (M) + 42:13 (F),"2,3,4,",F,AB,AB Heterozygote,1,TF-2581-10-e3_S67_L001_R1_001.fastq.gz,TF-2581-10-e3_S67_L001_R2_001.fastq.gz,NaN
ERR12383247,14794943760,PRJEB70958,SAMEA114860213,ERX11759665,Sample 1 females genotype AA homozygote,TF2581-11,11,13:20 (M) + 42:13 (F),"2,3,4,",F,AB,AB Heterozygote,1,TF-2581-11_S9_L001_R1_001.fastq.gz,TF-2581-11_S9_L001_R2_001.fastq.gz,NaN
ERR12383248,15610248328,PRJEB70958,SAMEA114860214,ERX11759666,Sample 2 females genotype AA homozygote,TF2581-12,12,13:20 (M) + 42:13 (F),"2,3,4,",F,AB,AB Heterozygote,1,TF-2581-12_S10_L001_R1_001.fastq.gz,TF-2581-12_S10_L001_R2_001.fastq.gz,NaN
ERR12383249,9412089720,PRJEB70958,SAMEA114860215,ERX11759667,Sample 3 females genotype AA homozygote,TF2581-13,13,42:13 (M) + 13:20 (F),"1,4,8",M,BA,AB Heterozygote,1,TF-2581-13_S11_L001_R1_001.fastq.gz,TF-2581-13_S11_L001_R2_001.fastq.gz,NaN
ERR12383250,9959631122,PRJEB70958,SAMEA114860216,ERX11759668,Sample 1 males genotype AA homozygote,TF2581-14-e2,14e 2,42:13 (M) + 13:20 (F),"1,4,8",M,BA,AB Heterozygote,1,TF-2581-14-e2_S68_L001_R1_001.fastq.gz,TF-2581-14-e2_S68_L001_R2_001.fastq.gz,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ERR12383313,27406244206,PRJEB70958,SAMEA114860279,ERX11759731,Sample 1 females genotype FF homozygote,TF2581-71,71,4:18 + 4:18,"2,6,9",F,FF,FF Homozygote,3,TF-2581-71_S59_L001_R1_001.fastq.gz,TF-2581-71_S59_L001_R2_001.fastq.gz,"Had TF ID: TF2581-70, and Sample ID: 71 before..."
ERR12383314,13458064958,PRJEB70958,SAMEA114860280,ERX11759732,Sample 2 females genotype FF homozygote,TF2581-72,72,4:18 + 4:18,"2,6,9",F,FF,FF Homozygote,3,TF-2581-72_S60_L001_R1_001.fastq.gz,TF-2581-72_S60_L001_R2_001.fastq.gz,"Had TF ID: TF2581-71, and Sample ID: 72 before..."
ERR12383315,11890141660,PRJEB70958,SAMEA114860281,ERX11759733,Sample 3 females genotype FF homozygote,TF2581-7,7,13:20 (M) + 42:13 (F),"2,3,4,",M,AB,AB Heterozygote,1,TF-2581-7_S7_L001_R1_001.fastq.gz,TF-2581-7_S7_L001_R2_001.fastq.gz,NaN
ERR12383316,13692674262,PRJEB70958,SAMEA114860282,ERX11759734,Sample 1 males genotype FF homozygote,TF2581-8,8,13:20 (M) + 42:13 (F),"2,3,4,",M,AB,AB Heterozygote,1,TF-2581-8_S8_L001_R1_001.fastq.gz,TF-2581-8_S8_L001_R2_001.fastq.gz,NaN


In [4]:
# create transpose
vst_t = norm_counts.T

pca = PCA(n_components=2)
pca_scores = pca.fit_transform(vst_t)

pca_df = metadata.copy()
pca_df['PC1'] = pca_scores[:,0]
pca_df['PC2'] = pca_scores[:,1]

# Explained variance
pc1_var = pca.explained_variance_ratio_[0] * 100
pc2_var = pca.explained_variance_ratio_[1] * 100

# Define your own color mapping
color_map = {'M': 'blue', 'F': 'red'}  

fig = px.scatter(
    pca_df,
    x='PC1',
    y='PC2',
    color='Sex',  
    color_discrete_map=color_map,          
    hover_name=pca_df.index,
    title=f"Salmon-Align: Male vs. Female PCA of VST-normalized counts"
)

fig.update_layout(
    xaxis_title=f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)",
    yaxis_title=f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)",
    plot_bgcolor="white",
        xaxis=dict(
        showgrid=True,
        gridcolor='lightgrey',
        gridwidth=1,
        zeroline=False
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor='lightgrey',
        gridwidth=1,
        zeroline=False
    ),
    width=1200,
    height=600,
)

fig.write_image("pca_salmon_align.svg")
fig.show()


Volcano PLot

In [5]:
results_full_annot = pd.read_csv("C:/Users/Sebas/OneDrive/Dokument/Master courses/MASTER THESIS/R project-Master Thesis/salmon_align_dominance_DE_sex_results_new_filtering_genotype_controlled_fixed_2.csv", float_precision='legacy')
results_full_annot

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,transcript_id,gene_id,seqname,chr_loc,...,COG_category,eggNOG_OGs,species_tree_birth_node,gene_tree_birth_node,birth_type,age_rank,node_depth_from_root,branch_length,n_copies_in_og,n_species_in_og
0,237.825076,0.769809,0.108163,7.117088,1.102309e-12,2.503789e-12,g2.t1,g2,utg000001l,A,...,I,"2FBM0@1|root,2TCUK@2759|Eukaryota,398E5@33154|...",C_maculatus_filtered_proteinfasta_TE_filtered,n3,duplication,9.0,0.758700,0.061748,2.0,8.0
1,73.844964,0.632045,0.112596,5.613409,1.983792e-08,3.880485e-08,g3.t1,g3,utg000001l,A,...,DUZ,"KOG2101@1|root,KOG2101@2759|Eukaryota,396Y9@33...",C_maculatus_filtered_proteinfasta_TE_filtered,n9,duplication,9.0,0.758700,0.061748,2.0,14.0
2,115.316375,-0.606464,0.077147,-7.861169,3.805645e-15,9.357857e-15,g4.t1,g4,utg000001l,A,...,H,"COG0181@1|root,KOG2892@2759|Eukaryota,38D6W@33...",C_maculatus_filtered_proteinfasta_TE_filtered,n13,duplication,9.0,0.758700,0.061748,2.0,14.0
3,251.432475,-0.863871,0.048939,-17.652036,9.815309e-70,7.588271e-69,g6.t1,g6,utg000001l,A,...,B,"2E6V3@1|root,2SDHR@2759|Eukaryota",N8,NaN,mrca_inferred,6.0,0.629385,0.171468,1.0,4.0
4,42.078202,-0.952676,0.102605,-9.284858,1.619236e-20,4.636125e-20,g7.t1,g7,utg000001l,A,...,Z,"COG4886@1|root,KOG0532@2759|Eukaryota,38HCP@33...",N0,NaN,mrca_inferred,1.0,0.000000,0.000000,1.0,12.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16558,11.656909,0.258990,0.195667,1.323623,1.856282e-01,2.200882e-01,g34843.t1,g34843,utg003648l,U,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16559,1.364347,2.030314,0.580010,3.500479,4.644220e-04,7.233045e-04,g34884.t1,g34884,utg003700l,U,...,S,"2D3G7@1|root,2SRFN@2759|Eukaryota,3AMW9@33154|...",C_maculatus_filtered_proteinfasta_TE_filtered,n17,duplication,9.0,0.758700,0.061748,23.0,4.0
16560,2.364626,0.345714,0.335347,1.030913,3.025814e-01,3.453507e-01,g34922.t1,g34922,utg003714l,U,...,NaN,NaN,C_maculatus_filtered_proteinfasta_TE_filtered,n43,duplication,9.0,0.758700,0.061748,22.0,6.0
16561,3.988477,1.095925,0.307969,3.558560,3.728934e-04,5.847220e-04,g35167.t1,g35167,utg003885l,U,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
# Replace the zeros in padj with 1e-308 avoid log10 issues
results_full_annot["padj_safe"] = results_full_annot["padj"].replace(0, 1e-308).fillna(1)

# Add the negative log 10 padj for plotting
results_full_annot["neglog10_padj"] = -np.log10(results_full_annot["padj_safe"])


# Add significance to differentially expressed transcripts 
results_full_annot["significant"] = (
    (results_full_annot["padj"] < 0.05) &
    (results_full_annot["log2FoldChange"].abs() > 1)
)

# Add a label to the significant transcripts
results_full_annot["label"] = results_full_annot["transcript_id"].where(results_full_annot["significant"], "")

results_full_annot

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,transcript_id,gene_id,seqname,chr_loc,...,birth_type,age_rank,node_depth_from_root,branch_length,n_copies_in_og,n_species_in_og,padj_safe,neglog10_padj,significant,label
0,237.825076,0.769809,0.108163,7.117088,1.102309e-12,2.503789e-12,g2.t1,g2,utg000001l,A,...,duplication,9.0,0.758700,0.061748,2.0,8.0,2.503789e-12,11.601402,False,
1,73.844964,0.632045,0.112596,5.613409,1.983792e-08,3.880485e-08,g3.t1,g3,utg000001l,A,...,duplication,9.0,0.758700,0.061748,2.0,14.0,3.880485e-08,7.411114,False,
2,115.316375,-0.606464,0.077147,-7.861169,3.805645e-15,9.357857e-15,g4.t1,g4,utg000001l,A,...,duplication,9.0,0.758700,0.061748,2.0,14.0,9.357857e-15,14.028824,False,
3,251.432475,-0.863871,0.048939,-17.652036,9.815309e-70,7.588271e-69,g6.t1,g6,utg000001l,A,...,mrca_inferred,6.0,0.629385,0.171468,1.0,4.0,7.588271e-69,68.119857,False,
4,42.078202,-0.952676,0.102605,-9.284858,1.619236e-20,4.636125e-20,g7.t1,g7,utg000001l,A,...,mrca_inferred,1.0,0.000000,0.000000,1.0,12.0,4.636125e-20,19.333845,False,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16558,11.656909,0.258990,0.195667,1.323623,1.856282e-01,2.200882e-01,g34843.t1,g34843,utg003648l,U,...,NaN,NaN,NaN,NaN,NaN,NaN,2.200882e-01,0.657403,False,
16559,1.364347,2.030314,0.580010,3.500479,4.644220e-04,7.233045e-04,g34884.t1,g34884,utg003700l,U,...,duplication,9.0,0.758700,0.061748,23.0,4.0,7.233045e-04,3.140679,True,g34884.t1
16560,2.364626,0.345714,0.335347,1.030913,3.025814e-01,3.453507e-01,g34922.t1,g34922,utg003714l,U,...,duplication,9.0,0.758700,0.061748,22.0,6.0,3.453507e-01,0.461740,False,
16561,3.988477,1.095925,0.307969,3.558560,3.728934e-04,5.847220e-04,g35167.t1,g35167,utg003885l,U,...,NaN,NaN,NaN,NaN,NaN,NaN,5.847220e-04,3.233051,True,g35167.t1


In [7]:
fig = px.scatter(
    results_full_annot,
    x="log2FoldChange",
    y="neglog10_padj",
    hover_name="transcript_id",
    hover_data=["padj", "HOG", "PFAMs"],  # Use the columns that already exist
    color="significant",
    color_discrete_map={True: "red", False: "blue"},
    title="Salmon-Align: Volcano Plot (Male vs. Female Transcript Expression)"
)

fig.update_traces(textposition='top center', textfont_size=8)

fig.update_layout(
    xaxis_title="log2 Fold Change (male vs female)",
    yaxis_title="-log10(padj (FDR))",
    width = 1200,
    height=600,
    plot_bgcolor="white",
    xaxis=dict(
        showgrid=True,
        gridcolor='lightgrey',
        gridwidth=1,
        zeroline=False
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor='lightgrey',
        gridwidth=1,
        zeroline=False
    ),
)

# Horizontal FDR=0.05 cutoff
padj_cut = -np.log10(0.05)

fig.add_hline(
    y=padj_cut,
    line_dash="dash",
    line_color="grey",
    annotation_text="padj = 0.05",
    annotation_position="bottom right"
)

# Vertical log2FC cutoffs
fig.add_vline(
    x=-1,
    line_dash="dash",
    line_color="grey",
    annotation_text="log2FC = -1",
    annotation_position="top left"
)
fig.add_vline(
    x=1,
    line_dash="dash",
    line_color="grey",
    annotation_text="log2FC = 1",
    annotation_position="top right"
)
fig.write_image("volcano_salmon_align.svg")
fig.show()


#DE genes
sig_counts = results_full_annot["significant"].sum()
higher_in_m = ((results_full_annot["significant"]) & 
               (results_full_annot["log2FoldChange"] > 0)).sum()
higher_in_f = ((results_full_annot["significant"]) & 
               (results_full_annot["log2FoldChange"] < 0)).sum()

print(f"Total significant transcripts: {sig_counts}")
print(f"Higher in males: {higher_in_m}")
print(f"Higher in females: {higher_in_f}")

Total significant transcripts: 6382
Higher in males: 4277
Higher in females: 2105
